# STAC AI Workshop · NYC · 21 October 2026

## 03 · What does training change?

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Compute SFT loss using a response-token mask
2. Explain LoRA within a selected projection
3. Compute DPO relative to a frozen reference
4. Separate a better objective from evidence support

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

In [1]:
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

Offline workbook ready: no API key, Node.js, GPU or model download.


## Same evidence as Web Section 3
Debt $75m, cash $15m, EBITDA $20m. **Signed covenant missing.** The reviewed answer requests the covenant rather than asserting comfortable headroom. All scores and tiny matrices below are declared illustrations; no model is trained.

## 1 · SFT: score the reviewed target, not the prompt
$L=-\sum_t m_t\log p(y_t\mid x,y_{<t})/\sum_t m_t$. Prompt mask 0; target mask 1. The three display units WITHHOLD / COVENANT / MISSING are illustrative tokens, not a claimed Qwen tokenization. Teacher forcing supplies the correct preceding answer tokens.

In [2]:
mask=[0,1,1,1]
initial=masked_nll([.9,.2,.1,.2],mask)
improved=masked_nll([.9,.7,.6,.7],mask)
show_table([{"Scores":"Initial","Mean target loss":round(initial,3)},{"Scores":"Higher target likelihood","Mean target loss":round(improved,3)}])
assert masked_nll([.01,.2,.1,.2],mask)==initial
assert improved<initial
print("The prompt score changes no loss because its mask is zero.")

Scores,Mean target loss
Initial,1.84
Higher target likelihood,0.408


The prompt score changes no loss because its mask is zero.


## 2 · LoRA: parallel trainable branch
$h=Wx+(\alpha/r)BAx$. W is frozen; A and B are trained using the chosen objective. LoRA adds parameters inside selected projections, not a new model between blocks.

In [3]:
show_table([{"Rank":r,"A dimensions":f"{r} × 896","B dimensions":f"896 × {r}","Added parameters":lora_count(896,896,r)} for r in (4,8,16)])
x=[2,1]; A=[1,-1]; B=[.5,-.5]; scale=2
base=x[:]  # W is identity
ax=sum(a*v for a,v in zip(A,x)); correction=[scale*b*ax for b in B]
output=[v+d for v,d in zip(base,correction)]
show_table([{"Quantity":"W x","Value":str(base)},{"Quantity":"A x","Value":ax},{"Quantity":"Scaled correction","Value":str(correction)},{"Quantity":"Output","Value":str(output)}])
assert output==[3,0]
base_count=494032768; adapter_count=(lora_count(896,896,8)+lora_count(896,128,8))*24
assert adapter_count==540672
print(f"Recorded earlier run: {base_count:,} + {adapter_count:,} = {base_count+adapter_count:,} parameters.")
print("The tiny matrices above were supplied, not fitted. Actual adapter size is a separate recorded fact.")

Rank,A dimensions,B dimensions,Added parameters
4,4 × 896,896 × 4,7168
8,8 × 896,896 × 8,14336
16,16 × 896,896 × 16,28672


Quantity,Value
W x,"[2, 1]"
A x,1
Scaled correction,"[1.0, -1.0]"
Output,"[3.0, 0.0]"


Recorded earlier run: 494,032,768 + 540,672 = 494,573,440 parameters.
The tiny matrices above were supplied, not fitted. Actual adapter size is a separate recorded fact.


## 3 · DPO: same prompt, preferred versus rejected response
Chosen: “Withhold; obtain the signed covenant.” Rejected: “Comfortable headroom; release.”

$z=\beta[(\log\pi(y^+)-\log\pi(y^-))-(\log\pi_{ref}(y^+)-\log\pi_{ref}(y^-))]$ and $L=-\log\sigma(z)$.
Scores are whole-response summed log probabilities. The reference remains (−8, −6), beta 0.1.

In [4]:
rows=[]
for label,c,r in [("Same as reference",-8,-6),("Favour chosen response",-5,-8),("Favour rejected response",-9,-5)]:
    value=dpo(c,r)
    rows.append({"Policy":label,"Chosen score":c,"Rejected score":r,"Margin":round(value["margin"],3),"DPO loss":round(value["loss"],3)})
show_table(rows)
assert dpo(-5,-8)["loss"]<dpo(-8,-6)["loss"]<dpo(-9,-5)["loss"]

Policy,Chosen score,Rejected score,Margin,DPO loss
Same as reference,-8,-6,0.0,0.693
Favour chosen response,-5,-8,0.5,0.474
Favour rejected response,-9,-5,-0.2,0.798


## 4 · Inspect evidence support
**Missing covenant → definition/ceiling unresolved → claim unsupported.** None of the arithmetic above changes that fact. Training on a preference is not an execution permission or calibrated probability of truth.

**Check:** what did lower loss establish? Higher likelihood of the declared target or improved agreement with the chosen preference—not factual correctness, generalization or alpha.

## Main takeaway
SFT/DPO choose what to reward; LoRA chooses how weights can change. The evidence graph still requires the missing document. Actual adaptation and historical replay are separate Section 4 exercises.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.